In [1]:
import numpy as np
from functools import reduce

In [2]:
# vocab size
V = 10

# embed size
E = 16

# head size
H = 5
num_heads = 3

In [3]:
# one hot vectors of tokens
one_hots = []
for i in range(V):
    one_hot = np.zeros((V,))
    one_hot[i] = 1
    one_hots.append(one_hot)
t = np.vstack(np.array(one_hots))
# 
assert t.shape == (V, V)

In [4]:
class Head:
    def __init__(self, W_e, W_u):
        self.W_e = W_e # E X V
        self.W_u = W_u # V X E
        self.W_q = np.random.rand(H,E)
        self.W_k = np.random.rand(H, E)
        self.W_v = np.random.rand(H, E)
        self.W_o = np.random.rand(H, E)
        #self.tril = np.tril(np.ones((V, V)))
        
    def softmax(self, x):
        return np.exp(x) / np.sum(np.exp(x), axis=1)
    
    def forward(self, t):
        assert t.shape == (V, V)
        QK = self.W_e.T @ self.W_q.T @ self.W_k @ self.W_e
        assert QK.shape == (V, V)
        # may need to do a masked attention
        #np.where(self.tril[:V, :V] == 0, float('-inf'), t.T @ QK @ t) 
        A_h = self.softmax(t.T @ QK @ t) -> # V X V 
        # V_d X V_s
        assert A_h.shape == (V, V)
        
        
        OV = self.W_u @ self.W_v.T @ self.W_o @ self.W_e
        # V_out X V_src
        assert OV.shape == (V, V)
        OV_out = OV @ t
        assert OV_out.shape == (V, V)
        
        return A_h @ OV_out
    

In [5]:
class Transformer:
    def __init__(self, num_heads):
        self.W_e = np.random.rand(E, V)
        self.W_u =  np.random.rand(V, E)
        self.heads = [Head(self.W_e, self.W_u) for _ in range(num_heads)]
        
    def forward(self, t):
        assert t.shape == (V, V)
        bigram = self.W_u @ self.W_e @ t
        attn_heads =  [head.forward(t) for head in self.heads]
        sum_attn_heads = reduce(lambda acc, attn_head: np.add(acc, attn_head), attn_heads)
        return bigram + sum_attn_heads
        

In [6]:
transformer = Transformer(num_heads)
transformer.forward(t)

array([[4.17142201e+04, 3.02078667e+04, 3.84761926e+04, 3.42040490e+04,
        3.26658427e+04, 3.82470674e+04, 3.75622341e+04, 3.47489171e+04,
        3.82622673e+04, 2.82736122e+04],
       [5.32280979e+00, 3.55152710e+00, 5.22848778e+00, 4.10551661e+00,
        4.75800364e+00, 4.55117168e+00, 4.47696491e+00, 3.69889738e+00,
        4.83529589e+00, 3.51616571e+00],
       [3.32051765e+01, 2.38496846e+01, 3.08043773e+01, 2.69074824e+01,
        2.54962852e+01, 3.06819491e+01, 2.90897364e+01, 2.72972457e+01,
        3.05274937e+01, 2.20778003e+01],
       [3.88150637e+00, 2.90031640e+00, 3.35162114e+00, 3.63096149e+00,
        3.70504742e+00, 3.35491373e+00, 4.45025446e+00, 3.58340296e+00,
        3.91823012e+00, 2.94033686e+00],
       [3.97976802e+00, 3.38324249e+00, 3.53741051e+00, 3.64922899e+00,
        3.78322178e+00, 3.34017055e+00, 3.98291897e+00, 3.89173932e+00,
        4.11232944e+00, 2.69428221e+00],
       [6.57433782e+02, 4.75285080e+02, 6.05210830e+02, 5.47222512e+02,
   